# Train Word2Vec

## Goal
Train a learned representation and generate document-level vectors.

## Required outputs
- vocabulary size
- training config
- saved embedding model path
- nearest-neighbor sanity checks

## Notes
- By default, the training script reads full processed splits.
- For a faster smoke run, pass max row overrides in the execution cell.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

# Resolve repository root regardless of whether kernel CWD is repo root or notebooks/.
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "src").exists() and (p / "configs").exists()),
    cwd,
)
PROJECT_ROOT

WindowsPath('C:/Coding/rep-learning-amazon-reviews')

In [2]:
import yaml

params = yaml.safe_load((PROJECT_ROOT / "params.yaml").read_text(encoding="utf-8"))
w2v_cfg = yaml.safe_load((PROJECT_ROOT / "configs" / "word2vec.yaml").read_text(encoding="utf-8"))

print("Dataset sample per category:", params["data"]["sample_per_category"])
print("Word2Vec config:")
print(json.dumps(w2v_cfg, indent=2))

Dataset sample per category: 500000
Word2Vec config:
{
  "name": "word2vec_skipgram",
  "architecture": "skipgram",
  "vector_size": 300,
  "window": 5,
  "min_count": 10,
  "negative": 10,
  "epochs": 10,
  "workers": 4,
  "seed": 42,
  "text_column": "cleaned_text",
  "min_review_tokens": 5,
  "max_review_tokens": 256,
  "vector_batch_size": 4096,
  "max_train_rows": null,
  "max_vector_rows": null,
  "sanity_words": [
    "battery",
    "charger",
    "kitchen",
    "beauty",
    "sports"
  ],
  "sanity_doc_pool_size": 5000,
  "sanity_top_k": 5,
  "document_pooling": "mean"
}


In [3]:
cmd = [
    sys.executable,
    "-m",
    "src.models.train_word2vec",
    # Uncomment to run a quick smoke test first:
    # "--max-train-rows", "120000",
    # "--max-vector-rows", "50000",
]

print("Running:", " ".join(cmd))
print("Working directory:", PROJECT_ROOT)
result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"Training command failed with exit code {result.returncode}")

Running: c:\Coding\rep-learning-amazon-reviews\.venv\Scripts\python.exe -m src.models.train_word2vec
Working directory: C:\Coding\rep-learning-amazon-reviews
Building Word2Vec vocabulary...
Vocabulary size: 32605
Training Word2Vec...
Phase 3 outputs saved:
- model: artifacts\models\word2vec_skipgram.model
- keyed vectors: artifacts\models\word2vec_skipgram.kv
- summary: artifacts\metrics\word2vec_skipgram_summary.json
- sanity checks: artifacts\metrics\word2vec_skipgram_sanity_checks.txt
- train vectors: artifacts\vectors\word2vec_skipgram_train_vectors.npy
- val vectors: artifacts\vectors\word2vec_skipgram_val_vectors.npy
- test vectors: artifacts\vectors\word2vec_skipgram_test_vectors.npy



In [4]:
summary_path = PROJECT_ROOT / "artifacts" / "metrics" / "word2vec_skipgram_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))

print("Vocabulary size:", summary["vocabulary_size"])
print("Model path:", summary["model_path"])
print("\nVector export coverage:")
for item in summary["split_vector_exports"]:
    print(f"- {item['split']}: rows={item['rows']} coverage={item['coverage']:.3f}")

print("\nWord-level sanity checks:")
for item in summary["word_sanity_checks"]:
    neighbors = ", ".join(
        f"{n['token']} ({n['score']:.3f})" for n in item["neighbors"][:3]
    )
    print(f"- {item['word']}: found={item['found']} | {neighbors}")

Vocabulary size: 32605
Model path: artifacts\models\word2vec_skipgram.model

Vector export coverage:
- train: rows=1395911 coverage=0.870
- val: rows=198948 coverage=0.867
- test: rows=405141 coverage=0.869

Word-level sanity checks:
- battery: found=True | batteries (0.711), batterie (0.594), batery (0.586)
- charger: found=True | charging (0.725), chargers (0.672), charge (0.614)
- kitchen: found=True | bathroom (0.668), countertop (0.544), counter (0.526)
- beauty: found=True | skincare (0.528), radha (0.495), blenders (0.476)
- sports: found=True | sport (0.586), sporting (0.515), baseball (0.508)


In [5]:
import pandas as pd

allowed = {
    "Electronics",
    "Home_and_Kitchen",
    "Beauty_and_Personal_Care",
    "Sports_and_Outdoors",
}

meta_paths = {
    "train": PROJECT_ROOT / "artifacts" / "vectors" / "word2vec_skipgram_train_meta.parquet",
    "val": PROJECT_ROOT / "artifacts" / "vectors" / "word2vec_skipgram_val_meta.parquet",
    "test": PROJECT_ROOT / "artifacts" / "vectors" / "word2vec_skipgram_test_meta.parquet",
}

for split, path in meta_paths.items():
    meta_df = pd.read_parquet(path)
    cats = set(meta_df["main_category"].astype(str).unique())
    unexpected = sorted(cats - allowed)
    print(f"{split} categories: {sorted(cats)}")
    if unexpected:
        raise ValueError(f"Unexpected categories in {split}: {unexpected}")

print("Category validation passed for all split metadata files.")

train categories: ['Beauty_and_Personal_Care', 'Electronics', 'Home_and_Kitchen', 'Sports_and_Outdoors']
val categories: ['Beauty_and_Personal_Care', 'Electronics', 'Home_and_Kitchen', 'Sports_and_Outdoors']
test categories: ['Beauty_and_Personal_Care', 'Electronics', 'Home_and_Kitchen', 'Sports_and_Outdoors']
Category validation passed for all split metadata files.


In [6]:
print("max_train_rows:", summary["config"].get("max_train_rows"))
print("max_vector_rows:", summary["config"].get("max_vector_rows"))

max_train_rows: None
max_vector_rows: None
